# V15 — Batch Size Study

## Purpose
Test whether collapse rate changes with batch size. CSD theorem is batch-size-independent.
Expected: BMIA stable across all batch sizes; entropy methods vary (smaller batch → noisier MI)

## Setup
1. Add dataset `rojanregmi1/cifar100-c` via Add Data
2. **Add dataset `cifar-100-python` via Add Data** ← avoids slow download
3. GPU T4 x2, Internet ON

## Design
- BATCH_SIZES: [16, 32, 64, 128, 256]
- Fixed: lr=0.05 (aggressive), severity=5, 5 corruptions, ResNet-18
- Methods: TENT, SAR, EATA, RDumb, BMIA-F, BMIA-A
- Eval always at batch=128 (fair comparison)
- Runs: 5 × 5 × 6 = 150
- Time: ~45 min

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import time
import json
import copy
import gc
import os

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

NUM_CLASSES = 100
EVAL_BATCH  = 128
N_SAMPLES   = 5000
LR          = 0.05
SEVERITY    = 5
BATCH_SIZES = [16, 32, 64, 128, 256]
CORRUPTIONS = ['gaussian_noise', 'defocus_blur', 'snow', 'contrast', 'jpeg_compression']
METHODS     = ['tent', 'sar', 'eata', 'rdumb', 'bmia_fixed', 'bmia_adaptive']

print(f'LR={LR}, BATCH_SIZES={BATCH_SIZES}, CORRUPTIONS={len(CORRUPTIONS)}')

In [ ]:
# ================================================================
# Step 1: Train ResNet-18 on CIFAR-100 (same as run/run, seed=42)
# ================================================================
import pickle
from PIL import Image
from torch.utils.data import Dataset

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

class CIFAR100Pickle(Dataset):
    """Load CIFAR-100 directly from pickle — works with any folder name."""
    def __init__(self, file_path, transform=None):
        with open(file_path, 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        self.imgs   = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.labels = np.array(d[b'fine_labels'], dtype=np.int64)
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = Image.fromarray(self.imgs[idx])
        if self.transform: img = self.transform(img)
        return img, int(self.labels[idx])

# ── Auto-detect CIFAR-100 pickle location ───────────────────────
CIFAR100_TRAIN_FILE = None
CIFAR100_TEST_FILE  = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'train' in files and 'test' in files and 'meta' in files:
        CIFAR100_TRAIN_FILE = os.path.join(root, 'train')
        CIFAR100_TEST_FILE  = os.path.join(root, 'test')
        print(f'Found CIFAR-100 at: {root}')
        break

if CIFAR100_TRAIN_FILE is None:
    print('Not found in /kaggle/input — downloading via torchvision...')
    trainset = torchvision.datasets.CIFAR100('/tmp/cifar100', train=True,
                                              download=True, transform=train_transform)
    testset  = torchvision.datasets.CIFAR100('/tmp/cifar100', train=False,
                                              download=True, transform=test_transform)
else:
    trainset = CIFAR100Pickle(CIFAR100_TRAIN_FILE, transform=train_transform)
    testset  = CIFAR100Pickle(CIFAR100_TEST_FILE,  transform=test_transform)
    print(f'Loaded: {len(trainset)} train, {len(testset)} test samples')
# ────────────────────────────────────────────────────────────────

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                           shuffle=True, num_workers=2)

model = torchvision.models.resnet18(weights=None)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, NUM_CLASSES)
model = model.to(device)

optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
criterion = nn.CrossEntropyLoss()

print('Training ResNet-18 (50 epochs)...')
for epoch in range(50):
    model.train()
    for X, y in trainloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}/50')

testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for X, y in testloader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(1) == y).sum().item()
        total   += y.size(0)
clean_acc = correct / total
print(f'\nClean accuracy: {clean_acc:.4f}')

SOURCE_STATE = copy.deepcopy(model.state_dict())
print('Source state saved.')

In [ ]:
# ================================================================
# Step 2: Load CIFAR-100-C
# ================================================================

DATA_PATH = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'labels.npy' in files and 'gaussian_noise.npy' in files:
        DATA_PATH = root
        break
assert DATA_PATH is not None, 'CIFAR-100-C not found! Add rojanregmi1/cifar100-c dataset.'
print(f'CIFAR-100-C at: {DATA_PATH}')

ALL_LABELS = np.load(os.path.join(DATA_PATH, 'labels.npy'))
CIFAR_MEAN = torch.tensor([0.5071, 0.4867, 0.4408]).view(1, 3, 1, 1)
CIFAR_STD  = torch.tensor([0.2675, 0.2565, 0.2761]).view(1, 3, 1, 1)

def load_corruption(name, severity=5, n_samples=N_SAMPLES):
    data = np.load(os.path.join(DATA_PATH, f'{name}.npy'))
    start = (severity - 1) * 10000
    imgs  = data[start:start + 10000][:n_samples]
    lbls  = ALL_LABELS[start:start + 10000][:n_samples]
    X = torch.from_numpy(imgs.copy()).permute(0, 3, 1, 2).float() / 255.0
    X = (X - CIFAR_MEAN) / CIFAR_STD
    return X.to(device), torch.from_numpy(lbls.copy()).long().to(device)

print('Data loading ready.')

In [ ]:
# ================================================================
# Step 3: Helpers
# ================================================================

def freeze_except_bn(mdl):
    bn_ids = set()
    for m in mdl.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for p in m.parameters():
                bn_ids.add(id(p))
    for p in mdl.parameters():
        p.requires_grad_(id(p) in bn_ids)

def get_metrics(mdl, X, y):
    mdl.eval()
    all_preds, all_probs = [], []
    with torch.no_grad():
        for i in range(0, X.size(0), EVAL_BATCH):
            logits = mdl(X[i:i+EVAL_BATCH])
            probs  = torch.softmax(logits, 1)
            all_preds.append(logits.argmax(1))
            all_probs.append(probs)
    preds  = torch.cat(all_preds)
    probs  = torch.cat(all_probs)
    acc    = (preds == y[:len(preds)]).float().mean().item()
    counts = torch.bincount(preds, minlength=NUM_CLASSES).float()
    dom    = counts.max().item() / counts.sum().item()
    n_cls  = (counts > 0).sum().item()
    h_yx   = -(probs * torch.log(probs + 1e-8)).sum(1).mean().item()
    mg     = probs.mean(0)
    h_y    = -(mg * torch.log(mg + 1e-8)).sum().item()
    return {'acc': round(acc, 4), 'dom_pct': round(dom, 4),
            'n_classes': n_cls, 'h_y': round(h_y, 4),
            'h_yx': round(h_yx, 4), 'mi': round(h_y - h_yx, 4)}

def is_collapsed(r):
    return (r['dom_pct'] > 0.15 and r['n_classes'] < 50) or r['n_classes'] < 20

print('Helpers ready.')

In [ ]:

# ================================================================
# Step 4: TTA Run Function (batch_size as parameter)
# ================================================================

def run_method(X, y, method, lr, batch_size):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train()
    freeze_except_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = optim.SGD(params, lr=lr, momentum=0.9)

    init_params = {pn: p.data.clone()
                   for pn, p in model.named_parameters() if p.requires_grad}
    current_lambda = 0.5
    batch_count    = 0
    fisher         = None

    if method == 'eata':
        # Fisher always uses EVAL_BATCH (128) so scaling is consistent across all batch sizes
        fisher = {pn: torch.zeros_like(p)
                  for pn, p in model.named_parameters() if p.requires_grad}
        actual_n = 0
        for j in range(0, min(128, X.size(0)), EVAL_BATCH):
            xb = X[j:j+EVAL_BATCH]
            if xb.size(0) < 2: continue
            model.zero_grad()
            probs = torch.softmax(model(xb), 1)
            (-(probs * torch.log(probs + 1e-8)).sum(1).mean()).backward()
            for pn, p in model.named_parameters():
                if pn in fisher and p.grad is not None:
                    fisher[pn] += p.grad.data ** 2 * xb.size(0)
            actual_n += xb.size(0)
        if actual_n > 0:
            for pn in fisher:
                fisher[pn] /= actual_n
        model.load_state_dict(copy.deepcopy(SOURCE_STATE))
        model.train()
        freeze_except_bn(model)
        params = [p for p in model.parameters() if p.requires_grad]
        opt    = optim.SGD(params, lr=lr, momentum=0.9)

    for i in range(0, X.size(0), batch_size):
        xb = X[i:i+batch_size]
        if xb.size(0) < 4:
            continue
        batch_count += 1

        if method == 'rdumb' and batch_count % 10 == 0:
            model.load_state_dict(copy.deepcopy(SOURCE_STATE))
            model.train()
            freeze_except_bn(model)
            params = [p for p in model.parameters() if p.requires_grad]
            opt    = optim.SGD(params, lr=lr, momentum=0.9)

        opt.zero_grad()
        probs = torch.softmax(model(xb), 1)
        ent   = -(probs * torch.log(probs + 1e-8)).sum(1)

        if method == 'tent':
            loss = ent.mean()
        elif method == 'rdumb':
            loss = ent.mean()
        elif method == 'sar':
            mask = ent < 0.4 * np.log(NUM_CLASSES)
            if mask.sum() < 2: continue
            loss = ent[mask].mean()
        elif method == 'eata':
            mask = ent < 0.4 * np.log(NUM_CLASSES)
            if mask.sum() < 2: continue
            fl   = sum((fisher[pn] * (p - init_params[pn])**2).sum()
                       for pn, p in model.named_parameters() if pn in fisher)
            loss = ent[mask].mean() + 2000 * fl
        elif method == 'bmia_fixed':
            mg   = probs.mean(0)
            loss = (ent.mean() - (-(mg * torch.log(mg + 1e-8)).sum()) +
                    1.0 * sum(((p - init_params[pn])**2).sum()
                              for pn, p in model.named_parameters() if pn in init_params))
        elif method == 'bmia_adaptive':
            mg   = probs.mean(0)
            loss = (ent.mean() - (-(mg * torch.log(mg + 1e-8)).sum()) +
                    current_lambda * sum(((p - init_params[pn])**2).sum()
                                         for pn, p in model.named_parameters()
                                         if pn in init_params))

        loss.backward()
        opt.step()

        if method == 'bmia_adaptive':
            with torch.no_grad():
                dom = (torch.bincount(model(xb).argmax(1),
                        minlength=NUM_CLASSES).float().max() / xb.size(0)).item()
            current_lambda = (min(10.0, current_lambda * 1.1) if dom > 0.10
                              else max(0.01, current_lambda * 0.95))

    return get_metrics(model, X, y)

print('run_method ready.')


In [ ]:
# ================================================================
# Step 5: Run All Experiments
# ================================================================

all_results = []
t_total = time.time()

for corr in CORRUPTIONS:
    X, y = load_corruption(corr, severity=SEVERITY)
    print(f'\n=== {corr} ===')

    for bs in BATCH_SIZES:
        for method in METHODS:
            t0 = time.time()
            r  = run_method(X, y, method=method, lr=LR, batch_size=bs)
            elapsed = time.time() - t0
            col = is_collapsed(r)
            mark = ' *** COLLAPSED ***' if col else ''
            print(f'  bs={bs:3d} {method:<14} acc={r["acc"]:.4f} '
                  f'dom={r["dom_pct"]:.1%} cls={r["n_classes"]:3d} '
                  f'MI={r["mi"]:.2f} ({elapsed:.0f}s){mark}')
            all_results.append({
                'corruption': corr, 'severity': SEVERITY,
                'method': method, 'lr': LR, 'batch_size': bs,
                **r, 'collapsed': col
            })
            gc.collect(); torch.cuda.empty_cache()

    del X, y; gc.collect(); torch.cuda.empty_cache()

print(f'\nTotal time: {(time.time()-t_total)/60:.1f} min')
print(f'Total results: {len(all_results)}')

In [ ]:
# ================================================================
# Step 6: Summary Tables
# ================================================================

print('='*80)
print('COLLAPSE RATE BY BATCH SIZE — CIFAR-100-C, sev=5, lr=0.05')
print('='*80)
print(f'\n{"Method":<16}', end='')
for bs in BATCH_SIZES:
    print(f' {"bs="+str(bs):>8}', end='')
print()
print('-'*65)
for method in METHODS:
    print(f'{method:<16}', end='')
    for bs in BATCH_SIZES:
        md = [r for r in all_results if r['method']==method and r['batch_size']==bs]
        nc = sum(1 for r in md if r['collapsed'])
        print(f' {nc}/{len(md)} ({nc/len(md)*100:.0f}%)' if md else f' {"N/A":>8}', end='')
    print()

print()
print('='*80)
print('ACCURACY BY BATCH SIZE')
print('='*80)
print(f'\n{"Method":<16}', end='')
for bs in BATCH_SIZES:
    print(f' {"bs="+str(bs):>9}', end='')
print()
print('-'*70)
for method in METHODS:
    print(f'{method:<16}', end='')
    for bs in BATCH_SIZES:
        md = [r for r in all_results if r['method']==method and r['batch_size']==bs]
        avg = np.mean([r['acc'] for r in md]) * 100 if md else 0
        print(f' {avg:>8.1f}%', end='')
    print()

In [ ]:
# ================================================================
# Step 7: Save + Print JSON
# ================================================================

save_data = {
    'experiment': 'V15_BATCH_SIZE',
    'model': 'ResNet-18 (CIFAR-100, seed=42)',
    'clean_acc': clean_acc,
    'fixed_lr': LR,
    'severity': SEVERITY,
    'batch_sizes': BATCH_SIZES,
    'corruptions': CORRUPTIONS,
    'results': all_results
}
with open('V15_BATCH_SIZE_results.json', 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print('Saved: V15_BATCH_SIZE_results.json')

with open('V15_BATCH_SIZE_results.json', 'r') as f:
    print(f.read())